# Bab 5: 5.11 Implementasi Python: Decision Tree & Pruning
**Tujuan:** mempraktikkan algoritma *decision tree* dalam tiga skenario, tanpa pruning, pre-pruning, dan post-pruning (*cost complexity pruning*), lalu membandingkan kinerjanya.

**Dataset:** Breast Cancer Wisconsin (`sklearn.datasets`, 569 pasien, 30 fitur numerik, 2 kelas).

## Praktikum 5.1: Persiapan Library
Tiga kelompok import: (1) komputasi & visualisasi, `numpy`, `pandas`, `matplotlib`; (2) data & split, `load_breast_cancer`, `train_test_split`, `validation_curve`; (3) model & evaluasi, `DecisionTreeClassifier`, `plot_tree`, `accuracy_score`, `classification_report`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

## Praktikum 5.2: Import Dataset
`as_frame=True` memuat data sebagai DataFrame agar nama fitur terbaca. Hasil: **569 baris × 30 fitur**, kelas 0 (*malignant*) = 212 sampel, kelas 1 (*benign*) = 357 sampel. Distribusi kelasnya tidak seimbang penuh tapi masih wajar untuk klasifikasi biner.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("Ukuran data:", X.shape)
print("Distribusi kelas:", np.bincount(y))

## Praktikum 5.3: Preprocessing (Split Data)
Data dibagi **80% latih / 20% uji**. `stratify=y` menjaga proporsi kedua kelas tetap sama di kedua belahan data (penting karena kelasnya tidak seimbang), dan `random_state=42` membuat hasil split selalu sama setiap dijalankan (*reproducible*).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.2, random_state=42, stratify=y)

## Praktikum 5.4: Training Model (dengan dan tanpa Pruning)
Tiga model dilatih untuk dibandingkan:
- **Tanpa pruning** (`tree_full`): pohon dibiarkan tumbuh sampai semua daun murni, kapasitas maksimal, risiko *overfitting*.
- **Pre-pruning** (`tree_pruned`): pertumbuhan dibatasi sejak awal, `max_depth=4` (kedalaman maks 4 level), `min_samples_leaf=5` (tiap daun minimal 5 sampel, mencegah daun berisi 1–2 sampel noise).
- **Post-pruning / CCP** (`tree_ccp`): pohon ditumbuhkan penuh dulu, lalu cabang lemah dipangkas memakai `cost_complexity_pruning_path`. Nilai `ccp_alpha` diambil dari tengah *pruning path* sebagai contoh; pada praktik idealnya dipilih via *validation curve* / *cross-validation*.

In [ ]:
# Model tanpa pruning (tumbuh penuh)
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Model dengan pre-pruning
tree_pruned = DecisionTreeClassifier(max_depth=4,
    min_samples_leaf=5, random_state=42)
tree_pruned.fit(X_train, y_train)

# Model dengan post-pruning (Cost Complexity Pruning)
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alpha_terpilih = path.ccp_alphas[len(path.ccp_alphas)//2]  # contoh pemilihan alpha
print("ccp_alpha terpilih:", ccp_alpha_terpilih)
tree_ccp = DecisionTreeClassifier(ccp_alpha=ccp_alpha_terpilih,
    random_state=42)
tree_ccp.fit(X_train, y_train)

## Praktikum 5.5: Evaluasi
Setiap model diukur akurasinya di data **latih** dan **uji**. **Gap** = akurasi train − akurasi test: gap besar = model menghafal data latih (*overfitting*), gap kecil = model menggeneralisasi dengan baik.

| Model | Kedalaman | Daun | Train | Test | Gap |
|---|---|---|---|---|---|
| Tanpa pruning | 7 | 19 | 1,000 | 0,912 | **0,088** |
| Pre-pruning | 4 | 10 | 0,976 | 0,921 | 0,055 |
| Post-pruning (CCP) | 4 | 7 | 0,978 | **0,930** | **0,048** |

**Bacaan tabel:** tanpa pruning mencapai train sempurna (1,000) tapi gap terbesar, pola khas *overfitting*. Pre-pruning memangkas gap ke 0,055. Post-pruning (CCP) terbaik: test tertinggi (0,930), gap terkecil (0,048), pohon paling ringkas (7 daun).

In [ ]:
for nama, model in [("Tanpa Pruning", tree_full),
    ("Pre-Pruning", tree_pruned), ("Post-Pruning (CCP)", tree_ccp)]:
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    print(f"{nama}: Akurasi Train={acc_train:.3f}, Akurasi Test={acc_test:.3f}, Gap={acc_train-acc_test:.3f}")

## Praktikum 5.6: Visualisasi
Pohon hasil pre-pruning digambar dengan `plot_tree`. Cara baca tiap node:
- **Baris 1:** fitur & ambang pemisah, mis. `worst radius <= 16.795`, sampel dengan nilai ≤ ambang ke kiri, selebihnya ke kanan.
- **gini:** ketidakmurnian node (0 = murni, makin kecil makin baik).
- **samples / value:** jumlah sampel dan perincian per kelas di node itu.
- **Warna:** biru = *benign*, oranye = *malignant*, makin pekat makin murni.

Pohon ini hanya sedalam 4 sehingga tiap jalur keputusan mudah dibaca sebagai aturan *if-then*, inilah keunggulan interpretabilitas *decision tree* dibanding model *black-box*.

In [ ]:
plt.figure(figsize=(14,8))
plot_tree(tree_pruned, feature_names=data.feature_names,
    class_names=data.target_names, filled=True, rounded=True,
    fontsize=7)
plt.title("Decision Tree (max_depth=4, Pre-Pruning)")
plt.show()

## 5.11.1: Interpretasi
Model tanpa pruning umumnya menunjukkan gap besar antara akurasi train dan test (indikasi *overfitting*), sedangkan model dengan pre-pruning atau post-pruning menunjukkan gap yang lebih kecil, meskipun akurasi train sedikit menurun. Pemilihan strategi pruning terbaik sebaiknya didasarkan pada *validation curve* atau *cross-validation*, bukan sekadar mencoba satu nilai parameter.

**Poin presentasi:**
1. *Overfitting* terlihat jelas: train 1,000 vs test 0,912 (gap 0,088) pada pohon penuh.
2. Pruning menukar sedikit akurasi train dengan generalisasi yang jauh lebih baik, CCP: test 0,930, gap 0,048.
3. Model sederhana (7 daun) mengalahkan model kompleks (19 daun) di data uji, prinsip *Occam's razor*.
4. Untuk pemilihan parameter yangcermat, gunakan *validation curve* / *cross-validation*, bukan satu nilai coba-coba.